# ⚠️ Python Exception Handling & Robust Error Management

> **Series:** Learn the Basics | **Focus:** Error Handling & Control Flow

Exceptions are events triggered during program execution that disrupt normal instruction flow. Mastering Python's exception handling mechanics allows developers to build fault-tolerant, self-healing, and production-grade applications.

---

## 📋 Table of Contents
1. [The Exception Hierarchy & Types](#1.-The-Exception-Hierarchy-&-Types)
2. [The Full `try / except / else / finally` Architecture](#2.-The-Full-try-/-except-/-else-/-finally-Architecture)
3. [Catching Multiple Exceptions](#3.-Catching-Multiple-Exceptions)
4. [Raising Exceptions & Exception Chaining (`raise ... from ...`)](#4.-Raising-Exceptions-&-Exception-Chaining)
5. [Custom Domain Exception Classes](#5.-Custom-Domain-Exception-Classes)
6. [Python 3.11+ Exception Groups (`ExceptionGroup` & `except*`)](#6.-Python-3.11+-Exception-Groups)
7. [The EAFP vs. LBYL Architectural Philosophies](#7.-The-EAFP-vs.-LBYL-Architectural-Philosophies)
8. [Hands-On Interactive Challenges](#8.-Hands-On-Interactive-Challenges)
9. [Quick Reference Card & Summary](#9.-Quick-Reference-Card-&-Summary)


---
## 1. The Exception Hierarchy & Types

All built-in exceptions inherit from `BaseException`. Standard application errors derive from `Exception`.


In [ ]:
# Common exceptions and inheritance
print("issubclass(ZeroDivisionError, ArithmeticError):", issubclass(ZeroDivisionError, ArithmeticError))
print("issubclass(ArithmeticError, Exception):         ", issubclass(ArithmeticError, Exception))
print("issubclass(Exception, BaseException):           ", issubclass(Exception, BaseException))


---
## 2. The Full `try / except / else / finally` Architecture

- `try`: Code that might raise an exception.
- `except`: Handles specific exceptions.
- `else`: Runs **only if no exception** was raised in `try`.
- `finally`: Guaranteed to run **always** (cleanup/resource release).


In [ ]:
def divide_numbers(a: float, b: float) -> float | None:
    result = None
    try:
        print(f"Attempting division: {a} / {b}")
        result = a / b
    except ZeroDivisionError as e:
        print(f"  [EXCEPT] Cannot divide by zero: {e}")
    else:
        print(f"  [ELSE] Division succeeded! Result = {result}")
    finally:
        print("  [FINALLY] Cleanup executed unconditionally.")
    return result

print("--- Case 1: Valid division ---")
divide_numbers(10, 2)

print("\n--- Case 2: Zero division ---")
divide_numbers(10, 0)


---
## 3. Catching Multiple Exceptions

Handle multiple distinct exceptions in a single tuple or through separate blocks.


In [ ]:
def parse_config_value(data: dict, key: str) -> int:
    try:
        raw_val = data[key]
        return int(raw_val)
    except (KeyError, ValueError, TypeError) as e:
        print(f"[Handled Error: {type(e).__name__}] {e}")
        return 0

print("Missing key: ", parse_config_value({}, "port"))
print("Invalid text:", parse_config_value({"port": "invalid_port"}, "port"))
print("Valid port:  ", parse_config_value({"port": "8080"}, "port"))


---
## 4. Raising Exceptions & Exception Chaining (`raise ... from ...`)

Exception chaining explicitly preserves the original root-cause exception traceback.


In [ ]:
class DatabaseConnectionError(Exception):
    pass

def connect_to_database(host: str):
    try:
        # Simulate socket connection error
        raise ConnectionRefusedError(f"Connection to {host}:5432 refused.")
    except ConnectionRefusedError as err:
        # Chain original error to domain exception
        raise DatabaseConnectionError("Failed to initialize database connection.") from err

try:
    connect_to_database("db.production.internal")
except DatabaseConnectionError as e:
    print(f"Caught domain error: {e}")
    print(f"Original root cause:  {e.__cause__}")


---
## 5. Custom Domain Exception Classes

Create domain-specific exceptions carrying rich structured metadata.


In [ ]:
class InsufficientFundsError(Exception):
    """Raised when an account withdrawal exceeds available balance."""
    def __init__(self, balance: float, requested: float):
        self.balance = balance
        self.requested = requested
        self.shortfall = requested - balance
        super().__init__(
            f"Insufficient funds: Balance is ${balance:,.2f}, but requested ${requested:,.2f} (Shortfall: ${self.shortfall:,.2f})"
        )

try:
    raise InsufficientFundsError(balance=50.0, requested=200.0)
except InsufficientFundsError as e:
    print(f"Error Message: {e}")
    print(f"Shortfall:     ${e.shortfall:,.2f}")


---
## 6. Python 3.11+ Exception Groups (`ExceptionGroup` & `except*`)

Introduced in Python 3.11 (PEP 654), `ExceptionGroup` allows raising and handling multiple concurrent errors (e.g. from async tasks).


In [ ]:
# ExceptionGroup demonstration
eg = ExceptionGroup("Multiple batch validation errors", [
    ValueError("Invalid age value"),
    TypeError("Expected string username"),
    ValueError("Negative price detected")
])

print(f"ExceptionGroup message: {eg.message}")
print(f"Contained exceptions:   {len(eg.exceptions)}")


---
## 7. The EAFP vs. LBYL Architectural Philosophies

- **EAFP**: *Easier to Ask for Forgiveness than Permission* (Pythonic standard: assume key exists, handle `KeyError` with `try/except`).
- **LBYL**: *Look Before You Leap* (Check `if key in d:` before accessing).


---
## 8. Hands-On Interactive Challenges


In [ ]:
# Challenge 1: Retry decorator with exception handling
def retry_operation(operation, max_retries: int = 3):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return operation()
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Operation failed after {max_retries} retries") from last_error

# Challenge 2: Safe dictionary deep navigation
def safe_get_nested(data: dict, *keys, default = None):
    curr = data
    try:
        for k in keys:
            curr = curr[k]
        return curr
    except (KeyError, TypeError, IndexError):
        return default

# Automated verification tests
attempts = 0
def flaky_func():
    global attempts
    attempts += 1
    if attempts < 2:
        raise ValueError("Temporary glitch")
    return "SUCCESS"

assert retry_operation(flaky_func) == "SUCCESS"

nested_data = {"user": {"profile": {"email": "test@example.com"}}}
assert safe_get_nested(nested_data, "user", "profile", "email") == "test@example.com"
assert safe_get_nested(nested_data, "user", "settings", "theme", default="dark") == "dark"

print("[OK] All Exception Challenges Passed!")


---
## 9. Quick Reference Card & Summary

| Structure | Purpose |
| :--- | :--- |
| `try:` | Executes code that may fail |
| `except ExceptionType as e:` | Handles specified exception |
| `else:` | Runs if no exception was raised in `try` |
| `finally:` | Runs unconditionally for cleanup |
| `raise MyError from root_err` | Explicit exception chaining |
